# Vibrational Modes — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the Vibrational Modes exercise section by section. Use it to **prototype your code** and **test your implementations** against the course library before submitting on the website.

Each section includes small tests you can use to check your work.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/vib_modes_companion.ipynb)

## Setup

In [2]:
%%capture
!pip install pymatgen
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [3]:
import numpy as np
import itertools
from typing import List

from symm4ml import groups, groups_fast, linalg, rep, vib_modes

### Reference data

These vertices, group tables, and irreps are used throughout the exercise for testing.
They are loaded from the course library (`symm4ml.vib_modes`), which uses `pymatgen` to generate symmetry operations.

In [4]:
# Vertex coordinates
tetrahedron = vib_modes.tetrahedron
octahedron = vib_modes.octahedron

# Group vector representations (3x3 rotation/reflection matrices)
Td_vec = vib_modes.Td_vec
Oh_vec = vib_modes.Oh_vec
C2v_vec = vib_modes.C2v_vec

# Multiplication tables
Td_table = vib_modes.Td_table
Oh_table = vib_modes.Oh_table
C2v_table = vib_modes.C2v_table

# Irreps (precomputed with fixed random seed)
Td_irreps = vib_modes.Td_irreps
C2v_irreps = vib_modes.C2v_irreps

# Oh_irreps is not stored in the module; compute it here
np.random.seed(42)
Oh_irreps = rep.infer_irreps(Oh_table)

print(f"Td: {len(Td_vec)} elements, {len(Td_irreps)} irreps")
print(f"Oh: {len(Oh_vec)} elements, {len(Oh_irreps)} irreps")
print(f"C2v: {len(C2v_vec)} elements, {len(C2v_irreps)} irreps")
print(f"Tetrahedron: {tetrahedron.shape[0]} vertices")
print(f"Octahedron: {octahedron.shape[0]} vertices")

Td: 24 elements, 5 irreps
Oh: 48 elements, 10 irreps
C2v: 4 elements, 4 irreps
Tetrahedron: 4 vertices
Octahedron: 6 vertices


---
## Section 1: Degrees of Freedom and Subspaces

A system of $N$ points in 3D has $3N$ degrees of freedom. We remove translational (3) and rotational (up to 3) degrees to isolate the vibrational modes.

### 1.1 `translation_projector(n, d=3)`

Return a projector for the translational degrees of freedom for $n$ vertices in $d$ dimensions.

**Hint:** The translation modes are uniform displacements along each spatial axis. Build an orthonormal basis for these and project.

In [ ]:
def translation_projector(n: int, d: int = 3) -> np.ndarray:
    """Projector of translation degrees of freedom for n objects in d dimensions.

    Input:
        n (int): Number of objects
        d (int): Dimensionality of the space (default: 3)
    Output:
        (n * 3, n * 3) np.array projector of translation degrees of freedom
    """
    # YOUR CODE HERE
    pass

In [5]:
vib_modes.translation_projector(4, 3)

array([[0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,
        0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,
        0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,
        0.25],
       [0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,
        0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,
        0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,
        0.25],
       [0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,
        0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,
        0.  ],
       [0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  ,
        0.25],
       [0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  ,
        0.  ],
       [0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25, 0.  , 0.  , 0.25,
        0.  ],
       [0.  , 0.  , 0

In [6]:
vib_modes.translation_projector(2, 7)

array([[0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. ,
        0. ],
       [0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. ,
        0. ],
       [0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. ,
        0. ],
       [0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. ,
        0. ],
       [0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. ,
        0. ],
       [0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5,
        0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. ,
        0.5],
       [0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. ,
        0. ],
       [0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. ,
        0. ],
       [0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. ,
        0. ],
       [0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. ,
        0. ],
       [0. , 0. , 0. , 0. , 0.5, 0. , 0. , 

In [8]:
print(vib_modes.translation_projector(2, 7).shape)
print(vib_modes.translation_projector(4, 3).shape)

(14, 14)
(12, 12)


In [ ]:
# No small tests in vib_modes.py for translation_projector.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    translation_projector(4, 3),
    vib_modes.translation_projector(4, 3),
    atol=1e-7
)
np.testing.assert_allclose(
    translation_projector(2, 7),
    vib_modes.translation_projector(2, 7),
    atol=1e-7
)
print("translation_projector comparison passed!")

### 1.2 `rotation_projector(vertices)`

Return a projector for the rotational degrees of freedom for vertices in 3D. Compute cross products between three orthogonal rotation axes and each vertex's position relative to the center of mass (assumed at the origin).

In [ ]:
def rotation_projector(vertices, tol=1e-8):
    """Return projector of rotational degrees of freedom
    Inputs:
        vertices: (n, 3) np.array of vertices
        tol: tolerance for Gram-Schmidt (zero vector)
    Output:
        A (n * 3, n * 3) projector onto the span of rotational degrees of freedom
    """
    # YOUR CODE HERE
    pass

In [9]:
print(vib_modes.rotation_projector(tetrahedron).shape)
print(vib_modes.rotation_projector(octahedron).shape)

(12, 12)
(18, 18)


In [ ]:
# No small tests in vib_modes.py for rotation_projector.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    rotation_projector(tetrahedron),
    vib_modes.rotation_projector(tetrahedron),
    atol=1e-7
)
np.testing.assert_allclose(
    rotation_projector(octahedron),
    vib_modes.rotation_projector(octahedron),
    atol=1e-7
)
print("rotation_projector comparison passed!")

### 1.3 `vibration_projector(vertices)`

Construct a projector for the vibrational degrees of freedom by subtracting the translational and rotational projectors from the identity.

In [ ]:
def vibration_projector(vertices, tol=1e-8):
    """Build vibrational projector for vertices in 3D space.
    Inputs:
        vertices: (n, 3) vertices
        tol: tolerance for zero vector
    Output:
        (n * 3, n * 3) projector of vibrational modes
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for vibration_projector.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    vibration_projector(tetrahedron),
    vib_modes.vibration_projector(tetrahedron),
    atol=1e-7
)
np.testing.assert_allclose(
    vibration_projector(octahedron),
    vib_modes.vibration_projector(octahedron),
    atol=1e-7
)
print("vibration_projector comparison passed!")

In [10]:
print(vib_modes.vibration_projector(tetrahedron).shape)
print(vib_modes.vibration_projector(octahedron).shape)
vib_modes.vibration_projector(tetrahedron)

(12, 12)
(18, 18)


array([[ 5.00000000e-01,  1.25000000e-01, -1.25000000e-01,
         5.55111512e-17,  1.25000000e-01, -1.25000000e-01,
        -2.50000000e-01, -1.25000000e-01,  1.25000000e-01,
        -2.50000000e-01, -1.25000000e-01,  1.25000000e-01],
       [ 1.25000000e-01,  5.00000000e-01, -1.25000000e-01,
        -1.25000000e-01, -2.50000000e-01,  1.25000000e-01,
         1.25000000e-01,  5.55111512e-17, -1.25000000e-01,
        -1.25000000e-01, -2.50000000e-01,  1.25000000e-01],
       [-1.25000000e-01, -1.25000000e-01,  5.00000000e-01,
         1.25000000e-01,  1.25000000e-01, -2.50000000e-01,
         1.25000000e-01,  1.25000000e-01, -2.50000000e-01,
        -1.25000000e-01, -1.25000000e-01,  5.55111512e-17],
       [ 5.55111512e-17, -1.25000000e-01,  1.25000000e-01,
         5.00000000e-01, -1.25000000e-01,  1.25000000e-01,
        -2.50000000e-01,  1.25000000e-01, -1.25000000e-01,
        -2.50000000e-01,  1.25000000e-01, -1.25000000e-01],
       [ 1.25000000e-01, -2.50000000e-01,  1.2500000

---
## Section 2: Build Representations

The permutation representation encodes how symmetry operations permute the vertices. Combined with the vector representation and the vibrational projector, it yields the vibrational representation.

### 2.1 `permutation_representation(vertices, vec_rep)`

Given vertex coordinates and the group's 3D vector representation, return the permutation representation. Use a tolerance `eps` to match transformed vertices to original vertices.

In [ ]:
def permutation_representation(vertices, vec_rep, eps=1e-4):
    """Return permutation representation of vertices under G with given vector representation.
    Inputs:
        vertices: (n, d) np.array of vertices
        vec_rep: (|G|, d, d) np.array of group representation on d-dimensional vector
        eps: absolute tolerance for matching vertices
    Output:
        (|G|, n, n) np.array of permutation representation of G on n vertices
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for permutation_representation.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    permutation_representation(tetrahedron, Td_vec),
    vib_modes.permutation_representation(tetrahedron, Td_vec),
    atol=1e-7
)
np.testing.assert_allclose(
    permutation_representation(octahedron, Oh_vec),
    vib_modes.permutation_representation(octahedron, Oh_vec),
    atol=1e-7
)
print("permutation_representation comparison passed!")

In [12]:
print(vib_modes.permutation_representation(octahedron, Td_vec).shape)
vib_modes.permutation_representation(tetrahedron, Td_vec)

(24, 6, 6)


array([[[0, 0, 1, 0],
        [0, 0, 0, 1],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[0, 1, 0, 0],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0]],

       [[0

In [11]:
print(vib_modes.permutation_representation(tetrahedron, Td_vec).shape)
vib_modes.permutation_representation(tetrahedron, Td_vec)

(24, 4, 4)


array([[[0, 0, 1, 0],
        [0, 0, 0, 1],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 1, 0],
        [0, 0, 0, 1]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0]],

       [[0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]],

       [[1, 0, 0, 0],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]],

       [[0, 1, 0, 0],
        [0, 0, 1, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0],
        [0, 0, 0, 1]],

       [[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 1, 0]],

       [[0, 0, 0, 1],
        [0, 0, 1, 0],
        [0, 1, 0, 0],
        [1, 0, 0, 0]],

       [[0

### 2.2 `vibration_representation(vertices, vec_rep)`

Compute the group representation on the vibrational subspace by combining the permutation representation, the vector representation (via tensor product), and the vibrational projector.

In [ ]:
def vibration_representation(vertices, vec_rep, eps=1e-4, tol=1e-8):
    """Returns the group representation in the vibration subspace.
    Inputs:
        vertices: (n, 3) np.array of vertices
        vec_rep: (|G|, 3, 3) np.array of group representation on 3D vector
        eps: absolute tolerance for matching vertices
        tol: tolerance for zero vector
    Output:
        (|G|, n*3, n*3) np.array of vibration representation of G on n vertices in 3D space. Note that the flattened n*3 indices should be in the unflattened order (vertex index, spatial index).
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for vibration_representation.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    vibration_representation(tetrahedron, Td_vec),
    vib_modes.vibration_representation(tetrahedron, Td_vec),
    atol=1e-7
)
print("vibration_representation comparison passed!")

In [13]:
print(vib_modes.vibration_representation(tetrahedron, Td_vec).shape)
vib_modes.vibration_representation(tetrahedron, Td_vec)

(24, 12, 12)


array([[[ 2.50000000e-01, -1.25000000e-01, -1.25000000e-01, ...,
          0.00000000e+00,  1.25000000e-01,  1.25000000e-01],
        [-1.25000000e-01,  0.00000000e+00,  1.25000000e-01, ...,
          1.25000000e-01, -2.50000000e-01, -1.25000000e-01],
        [-1.25000000e-01,  1.25000000e-01,  2.50000000e-01, ...,
         -1.25000000e-01,  1.25000000e-01,  2.50000000e-01],
        ...,
        [ 6.93889390e-18,  1.25000000e-01, -1.25000000e-01, ...,
          2.50000000e-01, -1.25000000e-01,  1.25000000e-01],
        [ 1.25000000e-01, -2.50000000e-01,  1.25000000e-01, ...,
         -1.25000000e-01,  0.00000000e+00, -1.25000000e-01],
        [ 1.25000000e-01, -1.25000000e-01,  2.50000000e-01, ...,
          1.25000000e-01, -1.25000000e-01,  2.50000000e-01]],

       [[ 5.00000000e-01,  1.25000000e-01, -1.25000000e-01, ...,
         -2.50000000e-01, -1.25000000e-01,  1.25000000e-01],
        [ 1.25000000e-01,  5.00000000e-01, -1.25000000e-01, ...,
         -1.25000000e-01, -2.50000000e

---
## Section 3: Analyzing Vibrational Modes

### 3.1 `vibration_irrep_projectors(vertices, vec_rep, irreps)`

Decompose the vibrational projector into subspace projectors, each transforming according to an irreducible representation (irrep).

**Algorithm:**
1. Calculate the $3N \times 3N$ group representation on the vibrational subspace.
2. Infer a change of basis between each irrep and the vibrational representation.
3. Construct and stack the projectors corresponding to each change of basis.
4. Return a real array if the imaginary components are below `tol`.

In [ ]:
def vibration_irrep_projectors(vertices, vec_rep, irreps, eps=1e-4, tol=1e-8):
    """Returns the vibrational projector decomposed into subspace projectors that transform according to each irreducible representation (irrep).
    Inputs:
        vertices: (n, 3) np.array of vertices
        vec_rep: (|G|, 3, 3) np.array of group representation on 3D vector
        irreps:
        eps: absolute tolerance for matching vertices
        tol: tolerance for zero vector
    Output:
        np.array of irrep projectors of shape (num_irreps, n * 3, n * 3)
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for vibration_irrep_projectors.
# Supplementary check (not a course small test):
projs = vibration_irrep_projectors(tetrahedron, Td_vec, Td_irreps)
# Each projector should be idempotent (P^2 = P)
for i, p in enumerate(projs):
    np.testing.assert_allclose(p @ p, p, atol=1e-7, err_msg=f"Projector {i} not idempotent")
# Projectors should sum to the vibration projector
np.testing.assert_allclose(
    projs.sum(axis=0),
    vib_modes.vibration_projector(tetrahedron),
    atol=1e-7,
)
print("vibration_irrep_projectors check passed!")

### 3.2 Which irreps?

Use `vib_modes.vibration_irrep_projectors`, `rep.character_table`, and the provided irreps to answer questions about the vibrational modes of a tetrahedron and an octahedron.

For reference, check the conventional character tables:
- [$T_d$](http://symmetry.jacobs-university.de/cgi-bin/group.cgi?group=902&option=4)
- [$O_h$](http://symmetry.constructor.university/cgi-bin/group.cgi?group=904&option=4)

The helper `vib_modes.conjugacy_class_in_table_notation` converts between 3D vector representations and standard notation for symmetry operations (e.g., $C_n$, $\sigma$, $S_n$, $E$, $i$).

In [ ]:
# Explore which irreps the vibrational modes transform as.
# Compute vibration_irrep_projectors for the tetrahedron and octahedron,
# then check which projectors have nonzero rank.
#
# Useful: np.trace(projector) gives the rank (= dimension of the subspace).
# Also try: rep.character_table(irreps, groups.conjugacy_classes(table))
# and vib_modes.conjugacy_class_in_table_notation(vec_rep, conj_classes)

# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWER: Which irreps do the vibrational modes of a tetrahedron transform as?
# Options: A_1, A_2, E, T_1, T_2
# tetra_mode_irreps = [...]  # list of irrep labels

In [ ]:
# YOUR ANSWER: Which irreps do the vibrational modes of an octahedron transform as?
# Options: A_{1g}, A_{2g}, E_g, T_{1g}, T_{2g}, A_{1u}, A_{2u}, E_u, T_{1u}, T_{2u}
# oct_mode_irreps = [...]  # list of irrep labels

### 3.3 Infrared and Raman Activity

**Infrared activity** arises from changes in the dipole moment, which transforms as a 3D vector $(x, y, z)$.

**Raman activity** is related to changes in polarizability, which transforms as the symmetric part of a $3 \times 3$ matrix: $(xy, yz, zx, 2z^2 - x^2 - y^2, x^2 - y^2)$.

Use the character table to determine which irreps are IR-active and Raman-active.

In [ ]:
# Explore IR and Raman activity.
# An irrep is IR-active if the tensor product of the irrep with the
# vector representation contains the trivial representation.
# An irrep is Raman-active if the tensor product with the symmetric
# rank-2 tensor representation contains the trivial representation.

# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWER: Which Td irreps are infrared active?
# Options: A_1, A_2, E, T_1, T_2
# tetra_ir_modes = [...]

In [ ]:
# YOUR ANSWER: Which Td irreps are Raman active?
# Options: A_1, A_2, E, T_1, T_2
# tetra_raman_modes = [...]

In [ ]:
# YOUR ANSWER: Which Oh irreps are infrared active?
# Options: A_{1g}, A_{2g}, E_g, T_{1g}, T_{2g}, A_{1u}, A_{2u}, E_u, T_{1u}, T_{2u}
# oct_ir_modes = [...]

In [ ]:
# YOUR ANSWER: Which Oh irreps are Raman active?
# Options: A_{1g}, A_{2g}, E_g, T_{1g}, T_{2g}, A_{1u}, A_{2u}, E_u, T_{1u}, T_{2u}
# oct_raman_modes = [...]

### 3.4 `vibrational_modes(vertices, vec_rep, irreps)`

Take the projectors from `vibration_irrep_projectors` and return an orthogonal basis for each irrep subspace. Each projector may contain multiple copies of an irrep — its rank is $n_i \times d_i$ where $n_i$ is the multiplicity and $d_i$ is the irrep dimension.

You can visualize modes with `vib_modes.plot_vibrational_mode(vertices, mode.reshape(-1, 3))`.

In [ ]:
def vibrational_modes(vertices, vec_rep, irreps, eps=1e-4, tol=1e-8):
    """Returns an orthogonal basis for each vibration subspace projectors that transform as specific irreps.
    Inputs:
        vertices: (n, 3) np.array of vertices
        vec_rep: (|G|, 3, 3) np.array of group representation on 3D vector
        irreps:
        eps: absolute tolerance for matching vertices
        tol: tolerance for zero vector
    Output:
        list of np.arrays with shape (irrep proj rank, n * 3) that are orthogonal bases for each irrep subspace
    """
    # YOUR CODE HERE
    pass

In [14]:
vib_modes.vibration_irrep_projectors(tetrahedron, Td_vec, Td_irreps)

array([[[ 8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02, -8.33333333e-02, -8.33333333e-02],
        [ 8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02, -8.33333333e-02, -8.33333333e-02],
        [-8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02, -8.33333333e-02,
          8.33333333e-02,  8.33333333e-02,  8.33333333e-02],
        [ 8.33333333e-02,  8.33333333e-02, -8.33333333e-02,
          8.33333333e-02, -8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02,  8.33333333e-02,  8.33333333e-02,
         -8.33333333e-02, -8.33333333e-02, -8.33333333e-02],
        [-8.33333333e-02, -8.3333333

In [ ]:
# No small tests in vib_modes.py for vibrational_modes.
# Supplementary check (not a course small test):
modes = vibrational_modes(tetrahedron, Td_vec, Td_irreps)
projs_course = vib_modes.vibration_irrep_projectors(tetrahedron, Td_vec, Td_irreps)
# Each mode set should be orthonormal and span the same space as the projector
for i, (m, p) in enumerate(zip(modes, projs_course)):
    if len(m) > 0:
        np.testing.assert_allclose(m @ m.T, np.eye(len(m)), atol=1e-7,
            err_msg=f"Modes for irrep {i} not orthonormal")
        np.testing.assert_allclose(
            linalg.gram_schmidt(m)[1], p, atol=1e-7,
            err_msg=f"Modes for irrep {i} don't span the correct subspace")
print("vibrational_modes check passed!")

In [ ]:
# Visualize vibrational modes of the tetrahedron
# modes_Td = vib_modes.vibrational_modes(tetrahedron, Td_vec, Td_irreps)
# for i, m in enumerate(modes_Td):
#     for j, mode in enumerate(m):
#         vib_modes.plot_vibrational_mode(
#             tetrahedron, mode.reshape(-1, 3),
#             title=f"Irrep {i}, mode {j}"
#         ).show()

---
## Section 4: Preserving Subgroup Symmetry

When a system undergoes a phase transition or distortion, its symmetry often reduces to a subgroup. Branching rules describe how the irreps of the full group decompose into those of the subgroup.

### 4.1 `isomorphic_subgroups(group_table, subgroup_table)`

Find all subgroups of a group that are isomorphic to a given candidate subgroup. For each match, return the subset of group elements and the list of valid isomorphisms.

**Steps:**
1. Generate candidate subgroups with `groups_fast.generate_subgroups_dynamic_programming`
2. Filter by size, compute subtables with `groups.subgroup_table_from_group_table` and `groups.remap_to_minimal`
3. Find isomorphisms with `groups_fast.isomorphisms_generator_backtracking`

In [ ]:
def isomorphic_subgroups(group_table, subgroup_table):
    """Identify isomorphic subgroups between a group and a candidate subgroup.

    This function takes in a table representing the full group and a table for a candidate subgroup,
    and determines for each subgroup the subset of group elements that form it and a list of valid
    isomorphisms between the subgroup and the candidate subgroup.

    Inputs:
    group_table: multiplication table of group, np.array of shape (|G|, |G|)
    subgroup_table: multiplication table of subgroup, np.array of shape (|H|, |H|)

    Returns:
      a list of tuples. For each tuple:
            - The first element is list of the subset of elements from the group that form the subgroup.
            - The second element is a list of valid isomorphisms mapping this subgroup to the candidate subgroup.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in vib_modes.py for isomorphic_subgroups.
# Supplementary comparison with course (not a course small test):
result = isomorphic_subgroups(Td_table, C2v_table)
course_result = vib_modes.isomorphic_subgroups(Td_table, C2v_table)
# Compare the sets of subgroup element sets
result_sets = set(frozenset(elems) for elems, _ in result)
course_sets = set(frozenset(elems) for elems, _ in course_result)
assert result_sets == course_sets, f"Subgroup sets differ: got {len(result_sets)}, expected {len(course_sets)}"
# Compare number of isomorphisms per subgroup
result_iso_counts = sorted(len(isos) for _, isos in result)
course_iso_counts = sorted(len(isos) for _, isos in course_result)
assert result_iso_counts == course_iso_counts, f"Isomorphism counts differ"
print("isomorphic_subgroups comparison passed!")

### 4.2 `branching_change_of_basis(G_irreps, H_irreps, H_elements)`

Compute change-of-basis matrices between the irreps of group $G$ and subgroup $H$ according to the branching rules.

**Steps:**
1. For each G irrep, restrict it to the subgroup elements: `G_ir[H_elements]`
2. For each H irrep, compute the change of basis with `linalg.infer_change_of_basis`
3. Return a nested list: `result[i][j]` = change-of-basis from G irrep $i$ to H irrep $j$

In [ ]:
def branching_change_of_basis(G_irreps, H_irreps, H_elements, tol=1e-6):
    """
    Compute the change-of-basis matrices between the irreducible representations (irreps) of a group G
    and those of a subgroup H according to the branching rules.

    This function assumes:
      - G_irreps is a list of NumPy arrays, each of shape (|G|, d, d), representing an irrep of G
        in a particular basis.
      - H_irreps is a list of NumPy arrays, each of shape (|H|, d, d), representing the corresponding
        irreps of H in a chosen basis.
      - H_elements is an ordered collection (e.g., list or array) of elements from G that form the subgroup H,
        arranged in the same order as the H_irreps.

    The function returns a list of change-of-basis matrices that map the basis of each irrep of G
    to the corresponding basis in which the irrep of H is expressed.

    Inputs:
      G_irreps : list of np.ndarray
          A list of irreducible representation matrices of group G. Each array has shape (|G|, d, d), where d
          is the dimension of that irrep.
      H_irreps : list of np.ndarray
          A list of irreducible representation matrices of subgroup H. Each array has shape (|H|, d, d).
      H_elements : list or array-like
          An ordered collection of elements of G that form the subgroup H, matching the order of the H_irreps.

    Output:
      list of list of np.ndarray
          A nested list where each outer list entry corresponds to a G irrep, and each inner list contains the
          change-of-basis matrices that transform the basis of the corresponding G irrep into the bases of the H irreps.
    """
    # YOUR CODE HERE
    pass

In [16]:
# No small tests in vib_modes.py for branching_change_of_basis.
# Supplementary comparison with course (not a course small test):
# Use one C2v subgroup of Td for testing
_subs_elements = [10, 6, 1, 23]
result = branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)
course_result = vib_modes.branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)
for i, (r_row, c_row) in enumerate(zip(result, course_result)):
    for j, (r, c) in enumerate(zip(r_row, c_row)):
        assert r.shape == c.shape, (
            f"Shape mismatch for G_irrep {i}, H_irrep {j}: got {r.shape}, expected {c.shape}"
        )
print("branching_change_of_basis comparison passed!")

NameError: name 'branching_change_of_basis' is not defined

In [17]:
vib_modes.branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)

[[array([], shape=(0, 1, 1), dtype=complex128),
  array([], shape=(0, 1, 1), dtype=complex128),
  array([], shape=(0, 1, 1), dtype=complex128),
  array([[[1.+0.j]]])],
 [array([], shape=(0, 1, 2), dtype=complex128),
  array([[[0.78105795+0.j, 0.62445855+0.j]]]),
  array([], shape=(0, 1, 2), dtype=complex128),
  array([[[ 0.62445855+0.j, -0.78105795+0.j]]])],
 [array([], shape=(0, 1, 1), dtype=complex128),
  array([[[1.+0.j]]]),
  array([], shape=(0, 1, 1), dtype=complex128),
  array([], shape=(0, 1, 1), dtype=complex128)],
 [array([[[0.81042266+0.j, 0.55942708+0.j, 0.17394383+0.j]]]),
  array([], shape=(0, 1, 3), dtype=complex128),
  array([[[ 0.53386915+0.j, -0.82748249+0.j,  0.17394383+0.j]]]),
  array([[[ 0.24124436+0.j, -0.04810478+0.j, -0.96927142+0.j]]])],
 [array([[[ 0.34087031+0.00000000e+00j, -0.90365773+3.69242724e-16j,
           -0.25924917+1.79487017e-16j]]]),
  array([[[ 0.33929316+0.00000000e+00j, -0.13892592+4.71932291e-17j,
            0.93036538-6.10754701e-16j]]]),
 

### 4.3 Branching rules of $T_d$ irreps under $C_{2v}$

Use `vib_modes.branching_change_of_basis` to determine how each $T_d$ irrep decomposes under $C_{2v}$. A nonzero change-of-basis matrix for a given $(G\text{-irrep}, H\text{-irrep})$ pair means that $H$-irrep appears in the branching of that $G$-irrep.

In [ ]:
# Explore branching rules.
# 1. Use isomorphic_subgroups (or vib_modes.isomorphic_subgroups) to find
#    a C2v subgroup of Td
# 2. Use branching_change_of_basis (or vib_modes.branching_change_of_basis)
#    to compute the branching rules
# 3. Check which change-of-basis matrices are nonzero

# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWERS: How does each Td irrep branch under C2v?
# Options for each: A_1, A_2, B_1, B_2
#
# A1_to_C2v = [...]   # e.g. ["A_1"]
# A2_to_C2v = [...]
# E_to_C2v = [...]
# T1_to_C2v = [...]
# T2_to_C2v = [...]

---
## Explore Further

In [ ]:
# Try analyzing a different molecule or point group!

In [ ]:
# Visualize vibrational modes